# 🏠 回家作業：在真實的量子電腦上執行

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_GITHUB_ID/quantum-intro-course/blob/main/02_homework_real_qpu.ipynb)

課堂上我們在模擬器跑出了 Bell 態。這次要把**同一個電路**送到 IBM 位在雲端的**真實量子電腦（QPU）**上，看看真實世界的結果。

> ⚠️ 一樣先點選 **「檔案 → 在雲端硬碟中儲存副本」**，再開始執行。

### 📋 你需要準備

1. **IBM Quantum Platform 帳號（免費）**：到 https://quantum.cloud.ibm.com/ 註冊並登入。
2. **建立 instance**：依照網站引導，選擇免費的 **Open Plan**。每 28 天有 10 分鐘的 QPU 時間，這份作業只會用掉幾秒鐘。
3. **取得 API key（token）**：在 IBM Quantum Platform 首頁可以建立並複製你的 API key。

> 🔐 **API key 就像你的密碼**，不要貼在程式碼裡、不要截圖、不要分享給別人。
> 下面的程式會用隱藏輸入框請你貼上，它不會被存進筆記本裡。

---
## 步驟 1：安裝與環境檢查

In [ ]:
# 安裝 Qiskit（量子程式套件）、Aer（模擬器）、IBM Runtime（連線 IBM 量子電腦用）
# 約需 1～2 分鐘，看到最後沒有紅色錯誤訊息就代表成功
!pip install -q 'qiskit[visualization]' qiskit-aer qiskit-ibm-runtime

In [ ]:
import sys, warnings
warnings.filterwarnings("ignore")   # 隱藏套件更新提示，不影響執行結果

import qiskit, qiskit_aer, qiskit_ibm_runtime

from qiskit import QuantumCircuit
from qiskit.visualization import plot_histogram
from qiskit_aer import AerSimulator
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler

print("Python            :", sys.version.split()[0])
print("qiskit            :", qiskit.__version__)
print("qiskit-aer        :", qiskit_aer.__version__)
print("qiskit-ibm-runtime:", qiskit_ibm_runtime.__version__)
print("✅ 環境準備完成！")
print("（圖表的文字使用英文，是因為 Colab 預設字型不支援中文）")

from qiskit_ibm_runtime import QiskitRuntimeService

---
## 步驟 2：連線到 IBM Quantum

執行下一格後會出現輸入框，貼上你的 API key 再按 Enter。輸入時畫面不會顯示內容，這是正常的。

> 💡 Colab 的執行階段重新啟動後，帳號資訊會消失，到時重新執行這一格即可。

In [ ]:
from getpass import getpass

token = getpass("請貼上你的 IBM Quantum API key，然後按 Enter：")
QiskitRuntimeService.save_account(
    channel="ibm_quantum_platform",
    token=token,
    overwrite=True,
    set_as_default=True,
)
del token   # 用完立刻刪掉，不留在記憶體中

service = QiskitRuntimeService()
print("✅ 連線成功！")

### 選一台量子電腦

讓系統自動挑一台**目前最不忙**的真機。

In [ ]:
backend = service.least_busy(operational=True, simulator=False)
print("你將使用的量子電腦：", backend.name)
print("量子位元數：", backend.num_qubits)
print("排隊中的工作數：", backend.status().pending_jobs)

---
## 步驟 3：建立電路並送出

這就是課堂上的 Bell 態電路，一模一樣。

In [ ]:
bell = QuantumCircuit(2)   # 準備 2 個量子位元，一開始都是 0
bell.h(0)                  # H 閘：讓量子位元 0 進入疊加態
bell.cx(0, 1)              # CNOT 閘：以 0 為控制、1 為目標，讓兩者糾纏
bell.measure_all()         # 測量全部量子位元

bell.draw("mpl")

真機可能需要**排隊**（從幾秒到更久都有可能），所以這次分兩步：先**送出工作**、記下工作編號，再**取回結果**。

In [ ]:
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_bell = pm.run(bell)

sampler = Sampler(mode=backend)
job = sampler.run([isa_bell], shots=1000)

job_id = job.job_id()
print("已送出！工作編號（請記下來）：", job_id)
print("目前狀態：", job.status())

### 取回結果

執行下一格，程式會**等到結果出來**才結束。如果排隊很久，可以先去做別的事，結果出來後再回來。

> 💡 如果 Colab 中斷了也沒關係，工作還在 IBM 那邊跑。重新執行步驟 1、2 之後，
> 把下一格的 `job_id` 改成你剛才記下的編號，就能用 `service.job(job_id)` 取回結果。
> 也可以到 IBM Quantum Platform 的 [Workloads](https://quantum.cloud.ibm.com/workloads) 頁面查看工作狀態。

In [ ]:
# job_id = "貼上你的工作編號"   # ← Colab 中斷後重新取回時才需要這一行
job = service.job(job_id)
real_counts = job.result()[0].data.meas.get_counts()

print("真機結果：", real_counts)
plot_histogram(real_counts, title=f"Bell state on {backend.name}")

---
## 步驟 4：和模擬器比較

在模擬器上跑同一個電路，把兩個結果並排比較。

In [ ]:
sim = AerSimulator()
sim_pm = generate_preset_pass_manager(backend=sim, optimization_level=1)
sim_counts = Sampler(mode=sim).run([sim_pm.run(bell)], shots=1000).result()[0].data.meas.get_counts()

print("模擬器：", sim_counts)
print("真　機：", real_counts)
plot_histogram([sim_counts, real_counts], legend=["Simulator", f"Real QPU ({backend.name})"])

<details>
<summary>💡 為什麼結果有差異呢：參考說明</summary>

真機結果中，除了 `00` 和 `11`，通常還會出現少量 `01` 和 `10`。
這是因為真實的量子電腦會受到**雜訊**影響：閘操作不完美、測量偶爾讀錯，機器的校準狀態也會隨時間變化。
減少與修正這些錯誤（量子糾錯），正是現在全世界與台灣的研究團隊最努力突破的方向。

除了雜訊，使用真機還有其他實際的差別：需要排隊、可用時間有限，而且每台機器的特性都不一樣。
</details>

---
## 🧹（選做）清除帳號資訊

如果你用的是共用電腦，或之後不再使用，可以執行下一格刪除這個執行環境中儲存的帳號資訊。

In [ ]:
QiskitRuntimeService.delete_account()
print("已刪除儲存的帳號資訊")

---
🎉 **恭喜！你已經在真正的量子電腦上執行過程式了。**

想繼續學習，可以從 [IBM Quantum Learning](https://quantum.cloud.ibm.com/learning/en) 的課程開始。

<sub>本作業改寫自 IBM Quantum Learning 課程〈Build and run your first quantum program〉：
https://quantum.cloud.ibm.com/learning/en/courses/use-a-qc-today/build-and-run-your-first-quantum-program</sub>